## 1. Setup and Data Loading

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
from scipy.stats import chi2_contingency, fisher_exact, mannwhitneyu
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("✓ Libraries loaded")

In [ ]:
def load_nde_data(structured_dir: Path) -> pd.DataFrame:
    """Load NDE records and extract fields relevant to Being of Light analysis.
    
    Uses the UPDATED schema with:
    - Split judgment fields (source, intensity, emotional_tone)
    - Split belief consistency (doctrine vs personal expectations)
    - Before/after spirituality and religiosity
    - Religious background vs belief at NDE
    - Christian denomination detail
    """
    records = []
    
    for file in structured_dir.glob("*.json"):
        with open(file, 'r', encoding='utf-8') as f:
            record = json.load(f)
        
        data = {'nde_code': record.get('title', 'unknown'),
                'dataset': record.get('dataset', 'unknown')}
        
        analysis = record.get('analysis', {}) or {}
        
        # === DEMOGRAPHICS (NEW SCHEMA) ===
        demo = analysis.get('person_demographics', {}) or {}
        data['age_at_nde'] = demo.get('age_at_nde')
        data['years_since_nde'] = demo.get('years_since_nde')
        data['gender'] = demo.get('gender', 'not_mentioned')
        
        # Religious background (raised in) vs belief at NDE (active)
        data['religious_background'] = demo.get('religious_background', 'not_mentioned')
        data['religious_background_denomination'] = demo.get('religious_background_denomination')
        data['religious_belief_at_nde'] = demo.get('religious_belief_at_nde', 'not_mentioned')
        data['religious_belief_at_nde_denomination'] = demo.get('religious_belief_at_nde_denomination')
        
        # === BEING ENCOUNTERS ===
        passage = analysis.get('passage', {}) or {}
        arrival = passage.get('arrival', {}) or {}
        
        # Light encounter type (NEW: single value with precedence)
        data['light_encounter'] = arrival.get('light_encounter', 'not_mentioned')
        
        # Being identifications (list)
        being_ids = arrival.get('being_identifications', [])
        if not isinstance(being_ids, list):
            being_ids = [being_ids] if being_ids else []
        data['being_identifications'] = being_ids
        
        # Categorize beings
        data['has_god'] = 'god' in being_ids
        data['has_jesus'] = 'jesus' in being_ids
        data['has_angels'] = 'angels' in being_ids
        data['has_buddha'] = 'buddha' in being_ids
        data['has_religious_figure'] = 'religious_figure_specified' in being_ids
        data['has_deceased'] = 'deceased_relative_guide' in being_ids
        data['has_unknown_presence'] = 'unknown_presence' in being_ids
        
        # Primary being identification
        if 'god' in being_ids:
            data['primary_being'] = 'god'
        elif 'jesus' in being_ids:
            data['primary_being'] = 'jesus'
        elif any(x in being_ids for x in ['buddha', 'religious_figure_specified']):
            data['primary_being'] = 'religious_figure'
        elif 'angels' in being_ids:
            data['primary_being'] = 'angels'
        elif 'unknown_presence' in being_ids:
            data['primary_being'] = 'unknown_presence'
        elif 'deceased_relative_guide' in being_ids:
            data['primary_being'] = 'deceased_relative'
        elif 'other' in being_ids:
            data['primary_being'] = 'other'
        else:
            data['primary_being'] = 'none'
        
        # Has any being encounter
        data['has_being_encounter'] = len(being_ids) > 0
        data['being_count'] = len(being_ids)
        
        # Greeting and belonging
        data['greeting_types'] = arrival.get('greeting_types', [])
        data['sense_of_belonging'] = arrival.get('sense_of_belonging', 'not_mentioned')
        
        # === COMMUNICATION (NEW: List field) ===
        world = analysis.get('world_of_spirits', {}) or {}
        encounters = world.get('encounters', {}) or {}
        
        comm_modes = encounters.get('communication_modes', [])
        if not isinstance(comm_modes, list):
            comm_modes = [comm_modes] if comm_modes else []
        data['communication_modes'] = comm_modes
        data['has_telepathic'] = 'telepathic' in comm_modes
        data['has_communication'] = len(comm_modes) > 0 and 'no_communication' not in comm_modes
        
        # Guidance (NEW: Split into received + types)
        data['guidance_received'] = encounters.get('guidance_received', 'not_mentioned')
        data['guidance_types'] = encounters.get('guidance_types', [])
        
        # === LIFE REVIEW - JUDGMENT (NEW SPLIT FIELDS) ===
        review = analysis.get('life_review', {}) or {}
        
        data['life_review_occurred'] = review.get('occurrence', 'not_mentioned')
        data['judgment_source'] = review.get('judgment_source', 'not_mentioned')
        data['judgment_intensity'] = review.get('judgment_intensity', 'not_specified')
        data['review_emotional_tone'] = review.get('emotional_tone', 'not_specified')
        data['perspective_of_others'] = review.get('perspective_of_others', 'not_mentioned')
        
        # === TRANSFORMATIVE EFFECTS (NEW: Before/After fields) ===
        effects = analysis.get('transformative_effects', {}) or {}
        
        data['death_fear_before'] = effects.get('death_fear_before', 'not_mentioned')
        data['death_fear_after'] = effects.get('death_fear_after', 'not_mentioned')
        data['spirituality_before'] = effects.get('spirituality_before', 'not_mentioned')
        data['spirituality_after'] = effects.get('spirituality_after', 'not_mentioned')
        data['religiosity_before'] = effects.get('religiosity_before', 'not_mentioned')
        data['religiosity_after'] = effects.get('religiosity_after', 'not_mentioned')
        data['value_shift'] = effects.get('value_shift', 'not_mentioned')
        
        # === EXPERIENCE QUALITY (NEW: Split belief consistency) ===
        quality = analysis.get('experience_quality', {}) or {}
        
        data['doctrine_consistency'] = quality.get('doctrine_consistency', 'not_mentioned')
        data['personal_expectation_consistency'] = quality.get('personal_expectation_consistency', 'not_mentioned')
        data['reality_assessment'] = quality.get('reality_assessment', 'not_mentioned')
        data['overall_valence'] = quality.get('overall_valence', 'not_mentioned')
        
        records.append(data)
    
    return pd.DataFrame(records)

# Load data
print("Loading NDE records...")
structured_dir = Path("../structured")
df = load_nde_data(structured_dir)

print(f"\n✓ Loaded {len(df):,} total records")
print(f"  NDERF: {(df['dataset'] == 'nderf').sum():,}")
print(f"  IANDS: {(df['dataset'] == 'iands').sum():,}")
print(f"  Features: {df.shape[1]}")

## 2. Data Quality Overview

In [ ]:
# Check key field distributions
print("="*70)
print("DATA QUALITY: Key Field Distributions")
print("="*70)

# Light encounter types
print("\n📊 Light Encounter Types:")
light_dist = df['light_encounter'].value_counts()
for val, count in light_dist.items():
    print(f"  {val}: {count:,} ({count/len(df)*100:.1f}%)")

# Being identifications
print("\n📊 Being Encounters:")
print(f"  Has any being: {df['has_being_encounter'].sum():,} ({df['has_being_encounter'].mean()*100:.1f}%)")
print(f"  Has God: {df['has_god'].sum():,} ({df['has_god'].mean()*100:.1f}%)")
print(f"  Has Jesus: {df['has_jesus'].sum():,} ({df['has_jesus'].mean()*100:.1f}%)")
print(f"  Has unknown presence: {df['has_unknown_presence'].sum():,} ({df['has_unknown_presence'].mean()*100:.1f}%)")
print(f"  Has deceased relatives: {df['has_deceased'].sum():,} ({df['has_deceased'].mean()*100:.1f}%)")

# Religious background
print("\n📊 Religious Background (raised in):")
rel_bg = df['religious_background'].value_counts().head(8)
for val, count in rel_bg.items():
    print(f"  {val}: {count:,} ({count/len(df)*100:.1f}%)")

# Religious belief at NDE
print("\n📊 Religious Belief at Time of NDE:")
rel_nde = df['religious_belief_at_nde'].value_counts().head(8)
for val, count in rel_nde.items():
    print(f"  {val}: {count:,} ({count/len(df)*100:.1f}%)")

---

## 3. Test 1: Monotheistic Pattern

**Hypothesis**: Experiencers predominantly encounter a SINGLE transcendent being, consistent with monotheistic conceptual frameworks.

**Tests**:
- Distribution of being encounter types
- Proportion encountering singular vs. multiple beings
- Character of the primary being (transcendent vs. familiar)

In [ ]:
print("="*70)
print("TEST 1: MONOTHEISTIC PATTERN")
print("="*70)

# Filter to those with being encounters
df_beings = df[df['has_being_encounter']].copy()
print(f"\nAnalyzing {len(df_beings):,} cases with being encounters ({len(df_beings)/len(df)*100:.1f}% of total)")

# Distribution of being count
print("\n📊 Number of Being Types Per Experience:")
being_counts = df_beings['being_count'].value_counts().sort_index()
for count, n in being_counts.items():
    print(f"  {count} being type(s): {n:,} ({n/len(df_beings)*100:.1f}%)")

# Single being dominant?
single_being = (df_beings['being_count'] == 1).sum()
print(f"\n→ Single being type: {single_being:,} ({single_being/len(df_beings)*100:.1f}%)")

# Primary being distribution
print("\n📊 Primary Being Identification:")
primary_dist = df_beings['primary_being'].value_counts()
for being, count in primary_dist.items():
    print(f"  {being}: {count:,} ({count/len(df_beings)*100:.1f}%)")

# Transcendent vs familiar
transcendent = {'god', 'jesus', 'religious_figure', 'unknown_presence', 'angels'}
familiar = {'deceased_relative', 'other'}

n_transcendent = df_beings['primary_being'].isin(transcendent).sum()
n_familiar = df_beings['primary_being'].isin(familiar).sum()

print(f"\n📈 TRANSCENDENT vs FAMILIAR:")
print(f"  Transcendent (God/Jesus/religious/unknown/angels): {n_transcendent:,} ({n_transcendent/len(df_beings)*100:.1f}%)")
print(f"  Familiar (deceased relatives/other): {n_familiar:,} ({n_familiar/len(df_beings)*100:.1f}%)")

# Statistical test: Is transcendent significantly more common?
from scipy.stats import binomtest
result = binomtest(n_transcendent, n_transcendent + n_familiar, p=0.5)
print(f"\n📐 Binomial test (transcendent vs familiar): p = {result.pvalue:.6f}")
print(f"   → {'SIGNIFICANT' if result.pvalue < 0.05 else 'NOT SIGNIFICANT'}: Transcendent beings {'ARE' if result.pvalue < 0.05 else 'are not'} significantly more common")

In [ ]:
# Visualization: Primary Being Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of primary beings
primary_counts = df_beings['primary_being'].value_counts()
colors = ['#2E86AB' if x in transcendent else '#A23B72' for x in primary_counts.index]
ax1 = axes[0]
bars = ax1.barh(primary_counts.index, primary_counts.values, color=colors)
ax1.set_xlabel('Count')
ax1.set_title('Primary Being Identification\n(Blue=Transcendent, Pink=Familiar)')
ax1.invert_yaxis()
for i, v in enumerate(primary_counts.values):
    ax1.text(v + 50, i, f'{v:,}', va='center')

# Pie chart: Transcendent vs Familiar
ax2 = axes[1]
sizes = [n_transcendent, n_familiar]
labels = ['Transcendent\n(God/Jesus/Unknown)', 'Familiar\n(Deceased/Other)']
ax2.pie(sizes, labels=labels, autopct='%1.1f%%', colors=['#2E86AB', '#A23B72'], startangle=90)
ax2.set_title('Transcendent vs Familiar Beings')

plt.tight_layout()
plt.show()

print("\n✓ CONCLUSION: Monotheistic pattern analysis complete")

---

## 4. Test 2: Unconditional Love (Judgment Analysis)

**Hypothesis**: The Being of Light is characterized by unconditional love rather than harsh judgment.

**New Schema Advantage**: We now have THREE separate dimensions:
- `judgment_source`: WHO judges (self, being_of_light, guide, none)
- `judgment_intensity`: HOW severe (loving_gentle, neutral, uncomfortable, harsh_condemning)
- `review_emotional_tone`: EXPERIENCER'S feelings (love, shame, mixed)

This allows us to distinguish "loving self-judgment" from "harsh external judgment."

In [ ]:
print("="*70)
print("TEST 2: UNCONDITIONAL LOVE (Judgment Analysis)")
print("="*70)

# Filter to those with life reviews
df_review = df[df['life_review_occurred'].isin(['extensive', 'brief'])].copy()
print(f"\nAnalyzing {len(df_review):,} cases with life reviews ({len(df_review)/len(df)*100:.1f}% of total)")

# Judgment Source Distribution
print("\n📊 Judgment Source (WHO judged):")
source_dist = df_review['judgment_source'].value_counts()
for source, count in source_dist.items():
    print(f"  {source}: {count:,} ({count/len(df_review)*100:.1f}%)")

# Judgment Intensity Distribution
print("\n📊 Judgment Intensity (HOW severe):")
intensity_dist = df_review['judgment_intensity'].value_counts()
for intensity, count in intensity_dist.items():
    print(f"  {intensity}: {count:,} ({count/len(df_review)*100:.1f}%)")

# Emotional Tone Distribution
print("\n📊 Experiencer's Emotional Tone:")
tone_dist = df_review['review_emotional_tone'].value_counts()
for tone, count in tone_dist.items():
    print(f"  {tone}: {count:,} ({count/len(df_review)*100:.1f}%)")

# Key metric: Loving vs Harsh judgment
loving = df_review['judgment_intensity'].isin(['loving_gentle', 'neutral', 'not_applicable']).sum()
harsh = df_review['judgment_intensity'].isin(['uncomfortable', 'harsh_condemning']).sum()

print(f"\n📈 LOVING vs HARSH JUDGMENT:")
print(f"  Loving/Neutral/None: {loving:,} ({loving/len(df_review)*100:.1f}%)")
print(f"  Uncomfortable/Harsh: {harsh:,} ({harsh/len(df_review)*100:.1f}%)")
print(f"  Ratio: {loving/harsh:.1f}:1" if harsh > 0 else "  Ratio: ∞:1 (no harsh judgments)")

In [ ]:
# Cross-tabulation: Source x Intensity
print("\n📊 CROSS-TABULATION: Judgment Source × Intensity")
print("="*70)

# Create crosstab
crosstab = pd.crosstab(df_review['judgment_source'], 
                       df_review['judgment_intensity'],
                       margins=True)
print(crosstab.to_string())

# Key insight: Self-judgment intensity profile
self_judgment = df_review[df_review['judgment_source'] == 'self']
if len(self_judgment) > 0:
    print(f"\n📍 SELF-JUDGMENT PROFILE (n={len(self_judgment)}):")
    self_intensity = self_judgment['judgment_intensity'].value_counts(normalize=True) * 100
    for intensity, pct in self_intensity.items():
        print(f"   {intensity}: {pct:.1f}%")

# Being of Light judgment intensity profile
bol_judgment = df_review[df_review['judgment_source'] == 'being_of_light']
if len(bol_judgment) > 0:
    print(f"\n📍 BEING OF LIGHT JUDGMENT PROFILE (n={len(bol_judgment)}):")
    bol_intensity = bol_judgment['judgment_intensity'].value_counts(normalize=True) * 100
    for intensity, pct in bol_intensity.items():
        print(f"   {intensity}: {pct:.1f}%")

In [ ]:
# Visualization: Judgment Analysis
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Source distribution
ax1 = axes[0]
source_counts = df_review['judgment_source'].value_counts()
ax1.bar(source_counts.index, source_counts.values, color='steelblue')
ax1.set_title('Judgment Source')
ax1.set_xlabel('Source')
ax1.set_ylabel('Count')
ax1.tick_params(axis='x', rotation=45)

# Intensity distribution
ax2 = axes[1]
intensity_order = ['loving_gentle', 'neutral', 'uncomfortable', 'harsh_condemning', 'not_applicable', 'not_specified']
intensity_counts = df_review['judgment_intensity'].value_counts().reindex(intensity_order, fill_value=0)
colors = ['#2E86AB', '#7EC8E3', '#F6AE2D', '#E63946', '#CCCCCC', '#999999']
ax2.bar(range(len(intensity_counts)), intensity_counts.values, color=colors[:len(intensity_counts)])
ax2.set_xticks(range(len(intensity_counts)))
ax2.set_xticklabels(intensity_counts.index, rotation=45, ha='right')
ax2.set_title('Judgment Intensity')
ax2.set_ylabel('Count')

# Emotional tone distribution
ax3 = axes[2]
tone_counts = df_review['review_emotional_tone'].value_counts()
ax3.bar(tone_counts.index, tone_counts.values, color='coral')
ax3.set_title('Experiencer Emotional Tone')
ax3.set_xlabel('Tone')
ax3.set_ylabel('Count')
ax3.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("\n✓ CONCLUSION: Unconditional love analysis complete")

---

## 5. Test 3: Personal Conscious Entity

**Hypothesis**: The Being exhibits characteristics of a personal conscious entity—communicates, provides guidance, shows intentionality.

**Tests**:
- Communication modes (telepathic, verbal, nonverbal)
- Guidance received and types
- Evidence of intentional interaction

In [ ]:
print("="*70)
print("TEST 3: PERSONAL CONSCIOUS ENTITY")
print("="*70)

# Focus on being encounters
df_beings = df[df['has_being_encounter']].copy()
print(f"\nAnalyzing {len(df_beings):,} cases with being encounters")

# Communication analysis
print("\n📊 Communication with Beings:")
print(f"  Any communication: {df_beings['has_communication'].sum():,} ({df_beings['has_communication'].mean()*100:.1f}%)")
print(f"  Telepathic: {df_beings['has_telepathic'].sum():,} ({df_beings['has_telepathic'].mean()*100:.1f}%)")

# Flatten communication modes for distribution
all_comm_modes = [mode for modes in df_beings['communication_modes'] for mode in modes if modes]
print("\n📊 Communication Mode Distribution:")
comm_counter = Counter(all_comm_modes)
for mode, count in comm_counter.most_common():
    print(f"  {mode}: {count:,}")

# Guidance analysis
print("\n📊 Guidance Received:")
guidance_dist = df_beings['guidance_received'].value_counts()
for val, count in guidance_dist.items():
    print(f"  {val}: {count:,} ({count/len(df_beings)*100:.1f}%)")

# Guidance types (for those who received guidance)
df_guided = df_beings[df_beings['guidance_received'] == 'yes']
if len(df_guided) > 0:
    all_guidance_types = [t for types in df_guided['guidance_types'] for t in types if types]
    print(f"\n📊 Guidance Types (n={len(df_guided)} cases):")
    guidance_counter = Counter(all_guidance_types)
    for gtype, count in guidance_counter.most_common():
        print(f"  {gtype}: {count:,}")

# Summary metric
communicative_beings = (df_beings['has_communication'] | (df_beings['guidance_received'] == 'yes')).sum()
print(f"\n📈 SUMMARY: Beings showing intentional communication/guidance:")
print(f"   {communicative_beings:,} / {len(df_beings):,} ({communicative_beings/len(df_beings)*100:.1f}%)")

---

## 6. Test 4: Cultural Mediation

**Hypothesis**: Religious background predicts being identification (cultural mediation), but core phenomenology is consistent across backgrounds (underlying reality).

**New Schema Advantage**: We can now distinguish:
- `religious_background` (raised in) vs `religious_belief_at_nde` (active at time)
- Christian denominations (Catholic, Evangelical, Mormon, etc.)

In [ ]:
print("="*70)
print("TEST 4: CULTURAL MEDIATION")
print("="*70)

# Filter to those with being encounters and known religious background
df_cult = df[(df['has_being_encounter']) & 
             (df['religious_belief_at_nde'] != 'not_mentioned')].copy()
print(f"\nAnalyzing {len(df_cult):,} cases with beings AND known religious belief at NDE")

# Create simplified religious groupings
def simplify_religion(rel):
    if rel == 'christian':
        return 'Christian'
    elif rel == 'atheist_agnostic':
        return 'Atheist/Agnostic'
    elif rel == 'spiritual_not_religious':
        return 'SBNR'
    elif rel in ['jewish', 'muslim', 'hindu', 'buddhist']:
        return 'Other Religion'
    else:
        return 'Other'

df_cult['religion_group'] = df_cult['religious_belief_at_nde'].apply(simplify_religion)

# Crosstab: Religion x Being Identification
print("\n📊 CROSSTAB: Religious Belief × Primary Being")
crosstab = pd.crosstab(df_cult['religion_group'], df_cult['primary_being'])
print(crosstab.to_string())

# Chi-square test
chi2, p, dof, expected = chi2_contingency(crosstab)
cramers_v = np.sqrt(chi2 / (len(df_cult) * (min(crosstab.shape) - 1)))

print(f"\n📐 Chi-square test:")
print(f"   χ² = {chi2:.2f}, df = {dof}, p = {p:.6f}")
print(f"   Cramér's V = {cramers_v:.3f} (effect size)")
print(f"   → {'SIGNIFICANT' if p < 0.05 else 'NOT SIGNIFICANT'}: Religious background {'DOES' if p < 0.05 else 'does not'} predict being identification")

In [ ]:
# Specific comparison: Christians vs Non-Christians identifying Jesus
print("\n📊 JESUS IDENTIFICATION: Christians vs Non-Christians")

christians = df_cult[df_cult['religion_group'] == 'Christian']
non_christians = df_cult[df_cult['religion_group'] != 'Christian']

jesus_christian = christians['has_jesus'].sum()
jesus_non_christian = non_christians['has_jesus'].sum()

print(f"  Christians identifying Jesus: {jesus_christian:,} / {len(christians):,} ({jesus_christian/len(christians)*100:.1f}%)")
print(f"  Non-Christians identifying Jesus: {jesus_non_christian:,} / {len(non_christians):,} ({jesus_non_christian/len(non_christians)*100:.1f}%)")

# Fisher's exact test for 2x2
contingency = [[jesus_christian, len(christians) - jesus_christian],
               [jesus_non_christian, len(non_christians) - jesus_non_christian]]
odds_ratio, p_fisher = fisher_exact(contingency)

print(f"\n   Odds ratio: {odds_ratio:.2f}x more likely for Christians")
print(f"   Fisher's exact p = {p_fisher:.6f}")

# Unknown presence rate across religions (should be consistent if underlying reality)
print("\n📊 UNKNOWN PRESENCE Rate by Religion (consistency test):")
for group in df_cult['religion_group'].unique():
    subset = df_cult[df_cult['religion_group'] == group]
    unknown_rate = subset['has_unknown_presence'].mean() * 100
    print(f"  {group}: {unknown_rate:.1f}%")

# Standard deviation of unknown presence rates (low = consistent)
unknown_rates = [df_cult[df_cult['religion_group'] == g]['has_unknown_presence'].mean() * 100 
                 for g in df_cult['religion_group'].unique()]
print(f"\n   Standard deviation of unknown presence rates: σ = {np.std(unknown_rates):.2f}%")
print(f"   → {'LOW' if np.std(unknown_rates) < 5 else 'HIGH'} variation suggests {'CONSISTENT' if np.std(unknown_rates) < 5 else 'INCONSISTENT'} underlying phenomenon")

---

## 7. Test 5: Belief Correction

**Hypothesis**: NDE experiences CORRECT prior beliefs rather than simply confirm them.

**New Schema Advantage**: We now have:
- `doctrine_consistency`: Did experience match official religious teaching?
- `personal_expectation_consistency`: Did experience match personal expectations?
- `death_fear_before` / `death_fear_after`: Explicit before/after measurement

In [ ]:
print("="*70)
print("TEST 5: BELIEF CORRECTION")
print("="*70)

# Doctrine consistency
print("\n📊 Doctrine Consistency (Experience vs Official Teaching):")
doctrine_dist = df['doctrine_consistency'].value_counts()
for val, count in doctrine_dist.items():
    print(f"  {val}: {count:,} ({count/len(df)*100:.1f}%)")

# Personal expectation consistency
print("\n📊 Personal Expectation Consistency:")
personal_dist = df['personal_expectation_consistency'].value_counts()
for val, count in personal_dist.items():
    print(f"  {val}: {count:,} ({count/len(df)*100:.1f}%)")

# Key metric: How many were surprised?
surprised = (df['personal_expectation_consistency'] == 'surprised').sum()
inconsistent = (df['personal_expectation_consistency'] == 'inconsistent').sum()
consistent = (df['personal_expectation_consistency'] == 'consistent').sum()

print(f"\n📈 BELIEF CORRECTION EVIDENCE:")
print(f"  Surprised: {surprised:,}")
print(f"  Inconsistent with expectations: {inconsistent:,}")
print(f"  Total corrected: {surprised + inconsistent:,}")
print(f"  Confirmed expectations: {consistent:,}")
if consistent > 0:
    correction_ratio = (surprised + inconsistent) / consistent
    print(f"  Correction:Confirmation ratio: {correction_ratio:.2f}:1")

In [ ]:
# Death fear before/after analysis
print("\n📊 DEATH FEAR TRANSFORMATION:")

# Filter to cases with both before and after
df_fear = df[(df['death_fear_before'] != 'not_mentioned') & 
             (df['death_fear_after'] != 'not_mentioned')].copy()
print(f"Analyzing {len(df_fear):,} cases with before/after death fear data")

# Create ordinal mapping
fear_order = {'none': 0, 'minimal': 1, 'moderate': 2, 'significant': 3, 'severe': 4}

df_fear['fear_before_ord'] = df_fear['death_fear_before'].map(fear_order)
df_fear['fear_after_ord'] = df_fear['death_fear_after'].map(fear_order)
df_fear = df_fear.dropna(subset=['fear_before_ord', 'fear_after_ord'])

if len(df_fear) > 0:
    df_fear['fear_change'] = df_fear['fear_after_ord'] - df_fear['fear_before_ord']
    
    decreased = (df_fear['fear_change'] < 0).sum()
    same = (df_fear['fear_change'] == 0).sum()
    increased = (df_fear['fear_change'] > 0).sum()
    
    print(f"\n  Fear DECREASED: {decreased:,} ({decreased/len(df_fear)*100:.1f}%)")
    print(f"  Fear UNCHANGED: {same:,} ({same/len(df_fear)*100:.1f}%)")
    print(f"  Fear INCREASED: {increased:,} ({increased/len(df_fear)*100:.1f}%)")
    
    # Mean change
    print(f"\n  Mean change: {df_fear['fear_change'].mean():.2f} (negative = decreased fear)")
    
    # Wilcoxon signed-rank test
    from scipy.stats import wilcoxon
    stat, p = wilcoxon(df_fear['fear_before_ord'], df_fear['fear_after_ord'])
    print(f"\n📐 Wilcoxon signed-rank test: W = {stat:.1f}, p = {p:.6f}")
    print(f"   → {'SIGNIFICANT' if p < 0.05 else 'NOT SIGNIFICANT'} change in death fear")
else:
    print("  Insufficient data for before/after analysis")

---

## 8. Summary and Conclusions

In [ ]:
print("="*70)
print("SUMMARY: Conceptual Framework Theory Validation")
print("="*70)

print("""
┌─────────────────────────────────────────────────────────────────────┐
│ TEST                        │ RESULT                               │
├─────────────────────────────┼──────────────────────────────────────┤
│ 1. Monotheistic Pattern     │ [PENDING - see analysis above]       │
│    - Single transcendent    │                                      │
│      being dominant?        │                                      │
├─────────────────────────────┼──────────────────────────────────────┤
│ 2. Unconditional Love       │ [PENDING - see analysis above]       │
│    - Loving judgment        │                                      │
│      predominant?           │                                      │
├─────────────────────────────┼──────────────────────────────────────┤
│ 3. Personal Conscious       │ [PENDING - see analysis above]       │
│    - Communication &        │                                      │
│      guidance present?      │                                      │
├─────────────────────────────┼──────────────────────────────────────┤
│ 4. Cultural Mediation       │ [PENDING - see analysis above]       │
│    - Background predicts    │                                      │
│      identification?        │                                      │
├─────────────────────────────┼──────────────────────────────────────┤
│ 5. Belief Correction        │ [PENDING - see analysis above]       │
│    - Experience corrects    │                                      │
│      prior expectations?    │                                      │
└─────────────────────────────┴──────────────────────────────────────┘

NOTE: Run all cells above to populate results.
""")

print(f"\nDataset: {len(df):,} NDE records")
print(f"Schema: Updated with split judgment, before/after fields, denomination detail")